In [ ]:
import logging

import itertools
from IPython.display import Markdown

from utils.circuit_faults import analyse_heuristic
from library.circuitry import Circuitry
from library.compact_magic_state_cultivation import MagicStateCultivation
from library.common import Pauli
from library.qubit_array import QubitArray
from library.surface_code.patch import SurfaceCodePatch
import warnings

logging.basicConfig(level=logging.ERROR)
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
TARGET_DISTANCE = 7

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1
minimum_anchoring = 1 + int(TARGET_DISTANCE == 7)
TARGET_ANCHOR = (minimum_anchoring, minimum_anchoring)

In [ ]:
scenarios: dict[str, Circuitry] = dict()

## Circuit 0 : Injection

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=0, tpt_rounds=0, exp_rounds=0)

msc.steane.append_noiseless_measurement(circuitry)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Injected"] = circuitry

In [ ]:
scenarios["Injected"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 1 : Injection + Superdense

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")

msc.annotate_detectors(circuitry, sdc_rounds=1, tpt_rounds=0, exp_rounds=0)

msc.steane.annotate_stabilizers(circuitry, style="DIFFUSE", noiseless=True)
for stabilizer, color in itertools.product("XZ", "RGB"):
    circuitry.annotate_detector(f"STN:{stabilizer}{color}", postselected=True)

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Superdense"] = circuitry

In [ ]:
# scenarios["Superdense"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 2 : Injection + Superdense + Double-check

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=1, tpt_rounds=0, exp_rounds=0)

msc.steane.annotate_stabilizers(circuitry, style="DIFFUSE", noiseless=True)
for stabilizer, color in itertools.product("XZ", "RGB"):
    circuitry.annotate_detector(f"STN:{stabilizer}{color}", postselected=True)

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Double-check"] = circuitry

In [ ]:
# scenarios["Double-checked"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 3 : Injection + Superdense + Double-check + Teleportation

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)
msc.append_teleportation(circuitry, TELEPORT_ROUNDS)

msc.annotate_detectors(
    circuitry, sdc_rounds=1, tpt_rounds=TELEPORT_ROUNDS, exp_rounds=0
)

msc.source.annotate_stabilizers(circuitry, noiseless=True)
for stabilizer, index in itertools.product(
    "XZ", range((msc.source.distance**2 - 1) // 2)
):
    circuitry.annotate_detector(
        f"SRC:REC:{stabilizer}{index}", f"STB:{stabilizer}{index}"
    )

circuitry.append_observable(
    0,
    "Y_OBSERVABLE_TELEPORTED",
    msc.source.logical(Pauli.Y),
    *(f"JCT0:{mz}" for mz in ["Z0", "Z1", "Z2"]),
    *(f"STN:DST:{mx}" for mx in ["X1", "X5", "X6"]),
    noiseless=True,
)

scenarios["Teleported"] = circuitry

In [ ]:
# scenarios["Teleported"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Circuit 4 : Injection + Superdense + Double-check + Teleportation + Expansion

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)
msc.append_teleportation(circuitry, TELEPORT_ROUNDS)
exp_rounds = msc.append_expansion(circuitry)

msc.annotate_detectors(
    circuitry, sdc_rounds=1, tpt_rounds=TELEPORT_ROUNDS, exp_rounds=exp_rounds
)

msc.target.annotate_stabilizers(circuitry, noiseless=True)
for stabilizer, index in itertools.product(
    "XZ", range((msc.target.distance**2 - 1) // 2)
):
    circuitry.annotate_detector(
        f"TGT:EXP:R{exp_rounds - 1}:{stabilizer}{index}",
        f"STB:{stabilizer}{index}",
    )

circuitry.append_observable(
    0,
    "Y_OBSERVABLE_EXPANDED",
    msc.target.logical(Pauli.Y, offset=TARGET_DISTANCE - 5),
    *(f"JCT0:{mz}" for mz in ["Z0", "Z1", "Z2"]),
    *(f"STN:DST:{mx}" for mx in ["X1", "X5", "X6"]),
    noiseless=True,
)

scenarios["Expanded"] = circuitry

In [ ]:
# scenarios["Expanded"].as_stim(per=0.01, keep_polygons=False).diagram("timeline-svg")

## Summary

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    print(f"Point {index} - {scenario}")
    circuitry.detectors_report()

### Circuits

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    label = f"Point {index} - {scenario}"
    display(Markdown(f"[Open in Crumble ({label})]({circuitry.to_crumble_url()})"))

### Fault analysis (S-proxies)

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    label = f"Point {index} - {scenario}"
    analyse_heuristic(circuitry.as_stim(per=0.01), label)

## Simulation and sampling [S-proxies]

In [ ]:
from utils.simulation.plotting import plot_results
import utils.simulation.stim as stim_simulation

In [ ]:
FILEROOT = "generated/s-proxies/hirano-magic-state-cultivation-layout1-compact"
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    circuitry.to_file(FILEROOT + f".point{index}.{scenario.lower()}")

In [ ]:
# Drop the scenarios to ignore
EXCLUDE = ["Injected", "Superdense", "Double-checked", "Teleported", "Expanded"]

In [ ]:
title = r"Magic State Cultivation of $|\mathbf{S}\rangle$ [Corrected $\overline{\mathbf{Y}}$]"

In [ ]:
stim_simulation.sample(scenarios, title=title, label="Point", shots=1e6, correction=True, fontsize=10)

In [ ]:
# results = stim_simulation.simulate(
#     scenarios,
#     label="Point",
#     postselection=True,
#     shots=1e6,
#     minimal_noise=-7,
#     num_workers=7,
#     # exclude=EXCLUDE,
# )

In [ ]:
# plot_results(
#     results, title, figsize=(11, 4.5), postselection=True, estimate_distance=True
# )

## Simulation and sampling [T-gates]

In [ ]:
import utils.simulation.clifft as clifft_simulation

In [ ]:
title = r"Magic State Cultivation of $|\mathbf{T}\rangle$ [Corrected $\frac{\overline{\mathbf{X}}+\overline{\mathbf{Z}}}{\sqrt{2}}$]"
NON_CLIFFORD_REPLACEMENTS = {"T": "T", "T_DAG": "T_DAG"} # Mapping of tag -> gate
EXCLUDE = [ "Teleported", "Expanded" ]

In [ ]:
FILEROOT = "generated/t-gates/hirano-magic-state-cultivation-layout1-compact"
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    circuitry.to_file(FILEROOT + f".point{index}.{scenario.lower()}", proxy_replacements=NON_CLIFFORD_REPLACEMENTS)

In [ ]:
# Sample the outcomes of the circuits at various points
clifft_simulation.sample(
    scenarios,
    proxy_replacements=NON_CLIFFORD_REPLACEMENTS,
    title=title,
    label="Point",
    shots=1e6,
    correction=True,
    postselection=True,
    figsize=(16, 5),
    fontsize=8,
)

In [ ]:
results = clifft_simulation.simulate(
    scenarios,
    label="Point",
    postselection=True,
    exclude=EXCLUDE,
    proxy_replacements=NON_CLIFFORD_REPLACEMENTS,
)

In [ ]:
plot_results(
    results, title, figsize=(11, 4.5), postselection=True, estimate_distance=True
)